# Experiment 1B: Monte Carlo comparison

The main study generates 200 instances independently of previous checkpoints. Runtime scaling is separate from the main comparison. Exact Rectangular evaluation above four sources is skipped by the configured guard. Outputs are stored in `rose_rag_monte_carlo_outputs`; the configuration retains the original local project-path fallback.


## Environment


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)

## Model and solvers


In [ ]:

from __future__ import annotations

import itertools
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from scipy.optimize import linprog


Array = np.ndarray


@dataclass(frozen=True)
class SourceSpec:
    """Nominal binary source specification."""
    alpha: float  # P(E=1 | Y=1)
    beta: float   # P(E=1 | Y=0)
    cost: float
    name: str


def make_channel(alpha: float, beta: float) -> Array:
    """
    Return a binary evidence channel with rows Y={0,1} and columns E={0,1}.
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha and beta must lie in [0, 1].")
    return np.array(
        [
            [1.0 - beta, beta],
            [1.0 - alpha, alpha],
        ],
        dtype=float,
    )


def _deduplicate_channels(channels: Array, names: Sequence[str]) -> Tuple[Array, List[str]]:
    seen = set()
    kept_channels: List[Array] = []
    kept_names: List[str] = []

    for channel, name in zip(channels, names):
        key = tuple(np.round(channel.ravel(), 12))
        if key not in seen:
            seen.add(key)
            kept_channels.append(channel)
            kept_names.append(name)

    return np.stack(kept_channels), kept_names


def build_coupled_scenarios(
    sources: Sequence[SourceSpec],
    rho: float,
    eps: float = 1e-9,
) -> Tuple[Array, List[List[Tuple[float, float]]], Array, List[str]]:
    """
    Build a finite, coupled ambiguity set.

    The latent regimes shift all sources coherently. This deliberately creates
    dependence across source models so that the history-consistent and fully
    rectangular formulations need not coincide.

    Returns
    -------
    scenario_channels:
        Shape (n_scenarios, n_sources, 2, 2).
    source_projection_points:
        Per-source projected (alpha, beta) channel points for Rect-ROSE.
    nominal_channels:
        Shape (n_sources, 2, 2).
    scenario_names:
        Names of retained coupled regimes.
    """
    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    regimes = [
        (0.0, 0.0, "nominal"),
        (-rho, +rho, "pessimistic"),
        (+rho, -rho, "optimistic"),
        (-rho, -rho, "low-positive-rate"),
        (+rho, +rho, "high-positive-rate"),
    ]

    raw_scenarios: List[Array] = []
    names: List[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario = []
        for source in sources:
            alpha = float(np.clip(source.alpha + delta_alpha, eps, 1.0 - eps))
            beta = float(np.clip(source.beta + delta_beta, eps, 1.0 - eps))
            scenario.append(make_channel(alpha, beta))
        raw_scenarios.append(np.stack(scenario))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(np.stack(raw_scenarios), names)

    source_projection_points: List[List[Tuple[float, float]]] = []
    for source_index in range(len(sources)):
        points: List[Tuple[float, float]] = []
        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (float(channel[1, 1]), float(channel[0, 1]))
            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)
        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return scenario_channels, source_projection_points, nominal_channels, names


def stop_value_precise(p_y1: float, abstention_cost: float) -> Tuple[float, str]:
    risks = [p_y1, 1.0 - p_y1, abstention_cost]
    actions = ["answer_0", "answer_1", "abstain"]
    index = int(np.argmin(risks))
    return float(risks[index]), actions[index]


def nominal_dp(
    prior_y1: float,
    channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float]], Dict]:
    """Exact nominal dynamic program for binary sources."""

    n_sources = len(costs)
    memo: Dict = {}

    def recurse(omega: Array, remaining: Tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]

        total = float(omega.sum())
        p_y1 = float(omega[1] / total)
        stopping_value, stopping_action = stop_value_precise(
            p_y1, abstention_cost
        )

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            for evidence in (0, 1):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    child_value = recurse(next_omega, next_remaining)[0]
                    value += evidence_probability * child_value

            continuation_values[source_index] = value

            if value < best_value - 1e-12:
                best_value = value
                best_action = f"query_{source_index}"

        memo[key] = (best_value, best_action, continuation_values)
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_dp_from_state(
    omega: Array,
    channels: Array,
    costs: Array,
    abstention_cost: float,
    remaining: Tuple[int, ...],
) -> float:
    """Nominal continuation value from an unnormalized binary belief."""

    memo: Dict = {}

    def recurse(current_omega: Array, current_remaining: Tuple[int, ...]) -> float:
        key = (tuple(float(value) for value in current_omega), current_remaining)
        if key in memo:
            return memo[key]

        total = float(current_omega.sum())
        p_y1 = float(current_omega[1] / total)
        best_value, _ = stop_value_precise(p_y1, abstention_cost)

        for source_index in current_remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index
                for index in current_remaining
                if index != source_index
            )

            for evidence in (0, 1):
                next_omega = (
                    current_omega * channels[source_index, :, evidence]
                )
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    value += evidence_probability * recurse(
                        next_omega, next_remaining
                    )

            best_value = min(best_value, value)

        memo[key] = float(best_value)
        return memo[key]

    return recurse(np.asarray(omega, dtype=float), remaining)


def hcr_dp(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    History-Consistent Rectangular dynamic program over a finite scenario set.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0

        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]

        answer_0_risk = float(np.max(posterior_y1))
        answer_1_risk = float(np.max(1.0 - posterior_y1))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                )[0]
                for evidence in (0, 1)
            }

            scenario_expectations = np.zeros(n_scenarios, dtype=float)

            for evidence in (0, 1):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)

                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )

                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )

            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _deduplicate_beliefs(values: Iterable[float]) -> Tuple[float, ...]:
    return tuple(sorted(set(round(float(value), 12) for value in values)))


def rectangular_dp(
    prior_y1: float,
    source_projection_points: Sequence[Sequence[Tuple[float, float]]],
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    Fully rectangular outer approximation.

    The state retains the projected belief set but discards dependence between
    the belief and the source model that generated it.
    """

    n_sources = len(costs)
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in source_points]
        for source_points in source_projection_points
    ]

    memo: Dict = {}

    def recurse(belief_set: Tuple[float, ...], remaining: Tuple[int, ...]):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)

        if key in memo:
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        if not len(beliefs):
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        answer_0_risk = float(np.max(beliefs))
        answer_1_risk = float(np.max(1.0 - beliefs))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_belief_sets: Dict[int, Tuple[float, ...]] = {}

            for evidence in (0, 1):
                posteriors: List[float] = []

                for belief in beliefs:
                    for channel in channel_sets[source_index]:
                        denominator = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )

                        if denominator > 0.0:
                            posterior = (
                                belief * channel[1, evidence] / denominator
                            )
                            posteriors.append(float(posterior))

                child_belief_sets[evidence] = _deduplicate_beliefs(posteriors)

            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence], next_remaining
                )[0]
                for evidence in (0, 1)
            }

            worst_expectation = -np.inf

            for belief in beliefs:
                for channel in channel_sets[source_index]:
                    expectation = 0.0

                    for evidence in (0, 1):
                        evidence_probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += (
                            evidence_probability * child_values[evidence]
                        )

                    worst_expectation = max(
                        worst_expectation, expectation
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse((prior_y1,), tuple(range(n_sources)))
    return root, memo


def static_best_response(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    adversary_distribution: Array,
):
    """
    Best deterministic policy against a distribution over fixed source models.

    The recursion returns a per-scenario risk vector. At the root, this vector
    is the expected total loss of the policy under each fixed scenario.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)

        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)

        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )
            child_zero = recurse(
                history + ((source_index, 0),), next_remaining
            )
            child_one = recurse(
                history + ((source_index, 1),), next_remaining
            )

            risk_vector = (
                costs[source_index] * reach_probability
                + child_zero[1]
                + child_one[1]
            )

            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    (child_zero[2], child_one[2]),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]

        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        policy_tree = (action, children)

        memo[key] = (
            weighted_values[selected_index],
            risk_vector,
            policy_tree,
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _solve_restricted_adversary(risk_matrix: Array):
    """
    Solve max_mu min_p <risk_p, mu> for the current policy columns.
    """

    n_policies, n_scenarios = risk_matrix.shape

    # Variables: mu_1,...,mu_S,v. Minimize -v.
    objective = np.zeros(n_scenarios + 1)
    objective[-1] = -1.0

    inequality_matrix = np.zeros((n_policies, n_scenarios + 1))
    inequality_matrix[:, :n_scenarios] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies)

    equality_matrix = np.zeros((1, n_scenarios + 1))
    equality_matrix[0, :n_scenarios] = 1.0
    equality_rhs = np.array([1.0])

    bounds = [(0.0, None)] * n_scenarios + [(None, None)]

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            f"Restricted static game failed: {result.message}"
        )

    adversary_distribution = result.x[:n_scenarios]
    game_value = float(result.x[-1])

    return game_value, adversary_distribution


def static_double_oracle(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    tolerance: float = 1e-9,
    max_iterations: int = 500,
):
    """
    Solve the exact finite-scenario randomized Static-ROSE game.

    A double-oracle method avoids enumerating all deterministic policy trees.
    The minimizer's best response is obtained by dynamic programming.
    """

    n_scenarios = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_scenarios) / n_scenarios

    first_response, _ = static_best_response(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
    )

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)

        upper_bound, adversary_distribution = (
            _solve_restricted_adversary(risk_matrix)
        )

        best_response, _ = static_best_response(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
        )

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)

        convergence_history.append(
            {
                "iteration": iteration,
                "upper_bound": upper_bound,
                "lower_bound": lower_bound,
                "gap": gap,
                "n_policy_columns": len(policy_risk_vectors),
            }
        )

        if gap <= tolerance:
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        vector_key = tuple(np.round(best_response[1], 12))

        if vector_key in observed_vectors:
            # Numerical stalling at machine precision.
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])

    raise RuntimeError(
        "Static double-oracle solver reached its iteration limit."
    )


def _robust_stop_from_likelihood(
    likelihood: Array,
    abstention_cost: float,
) -> float:
    reach_probability = likelihood.sum(axis=1)
    active = reach_probability > 0.0

    if not np.any(active):
        return 0.0

    posterior_y1 = (
        likelihood[active, 1] / reach_probability[active]
    )

    return float(
        min(
            np.max(posterior_y1),
            np.max(1.0 - posterior_y1),
            abstention_cost,
        )
    )


def root_acquisition_certificates(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
):
    """
    Compute root-level perfect-information lower bounds and query-then-stop
    upper bounds for all source actions.
    """

    import pandas as pd

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    root_likelihood = np.tile(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        (n_scenarios, 1),
    )

    hcr_root, _ = hcr_dp(
        prior_y1, scenario_channels, costs, abstention_cost
    )
    stopping_value = float(min(hcr_root[3]))

    rows = []

    for source_index in range(n_sources):
        remaining = tuple(
            index for index in range(n_sources) if index != source_index
        )

        child_stopping_values = {}
        for evidence in (0, 1):
            next_likelihood = (
                root_likelihood
                * scenario_channels[:, source_index, :, evidence]
            )
            child_stopping_values[evidence] = (
                _robust_stop_from_likelihood(
                    next_likelihood, abstention_cost
                )
            )

        lower_scenario_values = []
        upper_scenario_values = []

        for scenario_index in range(n_scenarios):
            perfect_information_expectation = 0.0
            query_then_stop_expectation = 0.0

            for evidence in (0, 1):
                next_omega = (
                    root_likelihood[scenario_index]
                    * scenario_channels[
                        scenario_index, source_index, :, evidence
                    ]
                )
                evidence_probability = float(next_omega.sum())

                if evidence_probability > 0.0:
                    perfect_information_expectation += (
                        evidence_probability
                        * nominal_dp_from_state(
                            next_omega,
                            scenario_channels[scenario_index],
                            costs,
                            abstention_cost,
                            remaining,
                        )
                    )
                    query_then_stop_expectation += (
                        evidence_probability
                        * child_stopping_values[evidence]
                    )

            lower_scenario_values.append(
                costs[source_index] + perfect_information_expectation
            )
            upper_scenario_values.append(
                costs[source_index] + query_then_stop_expectation
            )

        lower_bound = float(max(lower_scenario_values))
        upper_bound = float(max(upper_scenario_values))
        exact_hcr_value = float(
            hcr_root[2][source_index]
        )

        rows.append(
            {
                "source_index": source_index,
                "LB": lower_bound,
                "C_HCR": exact_hcr_value,
                "UB": upper_bound,
                "stop_value": stopping_value,
                "bound_valid": (
                    lower_bound
                    <= exact_hcr_value + 1e-10
                    and exact_hcr_value
                    <= upper_bound + 1e-10
                ),
                "rejection_certificate": (
                    lower_bound >= stopping_value - 1e-12
                ),
                "no_harm_certificate": (
                    upper_bound < stopping_value - 1e-12
                ),
            }
        )

    return pd.DataFrame(rows)


def posterior_dilation_unit_test() -> Dict[str, float]:
    """
    Reproduce the analytical cost-free retrieval dilation example.
    """

    prior_y1 = 0.9
    abstention_cost = 0.4
    costs = np.array([0.0])

    channel_a = make_channel(alpha=0.9, beta=0.1)
    channel_b = make_channel(alpha=0.1, beta=0.9)

    scenarios = np.stack(
        [
            np.stack([channel_a]),
            np.stack([channel_b]),
        ]
    )

    root, _ = hcr_dp(
        prior_y1, scenarios, costs, abstention_cost
    )

    initial_stopping_value = float(min(root[3]))
    continuation_value = float(root[2][0])
    robust_value_of_evidence = (
        initial_stopping_value - continuation_value
    )

    expected = {
        "initial_stopping_value": 0.1,
        "post_retrieval_hcr_value": 0.4,
        "RVoE_HCR": -0.3,
    }

    actual = {
        "initial_stopping_value": initial_stopping_value,
        "post_retrieval_hcr_value": continuation_value,
        "RVoE_HCR": robust_value_of_evidence,
    }

    for key, expected_value in expected.items():
        if not np.isclose(actual[key], expected_value, atol=1e-10):
            raise AssertionError(
                f"Dilation unit test failed for {key}: "
                f"expected {expected_value}, obtained {actual[key]}."
            )

    return actual




## Configuration and experiment functions

`N_INSTANCES` controls the sample count. The recorded main setting is 200. Runtime scaling is disabled by default and has separate resource limits.


In [ ]:

# Experiment 1B: Monte Carlo Theorem Stress Test


from pathlib import Path
from typing import Any
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


import os
import signal
from contextlib import contextmanager


class SolverTimeoutError(TimeoutError):
    """Raised when a runtime-scaling solver exceeds its time budget."""


@contextmanager
def solver_time_limit(seconds: float | None):
    """
    Interrupt a long pure-Python/scipy solver call on Unix-like systems.

    The main Monte Carlo study is not subject to this timer. It is used
    only in the separate runtime-scaling experiment.
    """
    if (
        seconds is None
        or seconds <= 0
        or os.name == "nt"
    ):
        yield
        return

    previous_handler = signal.getsignal(signal.SIGALRM)

    def _handle_timeout(signum, frame):
        raise SolverTimeoutError(
            f"Solver exceeded {seconds:.1f} seconds."
        )

    signal.signal(signal.SIGALRM, _handle_timeout)
    signal.setitimer(signal.ITIMER_REAL, seconds)

    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, previous_handler)



# Configuration


SEED = 20260712

# Use 200--500 for the final paper run.
# A first diagnostic run can use 20--50.
N_INSTANCES = 200

RHO_GRID = np.array([0.00, 0.02, 0.05, 0.10, 0.15, 0.20], dtype=float)

# Certificates are evaluated at this ambiguity radius.
CERTIFICATE_RHO = 0.10

# Number of sources in the main Monte Carlo experiment.
N_SOURCES = 3

# Numerical tolerances.
ORDER_TOL = 1e-8
MONOTONICITY_TOL = 1e-8
STATIC_SOLVER_TOL = 1e-9

# Save intermediate CSVs every this many completed instances.
CHECKPOINT_EVERY = 10

# Runtime-scaling experiment.
RUN_RUNTIME_SCALING = False
RUNTIME_SOURCE_COUNTS = [2, 3, 4, 5]
RUNTIME_REPETITIONS = 5
RUNTIME_RHO = 0.10

# Exact Rect-ROSE has a combinatorial belief-set explosion. With the
# current ambiguity construction, each source can have 13 projected
# channel models at rho=0.10; five sources can therefore create up to
# 13^5 model combinations before history branching is considered.
RECT_EXACT_MAX_SOURCES = 4

# Maximum wall-clock time allocated to one runtime-scaling solver call.
# This guard is supported on Linux/macOS, including Vast.ai.
RUNTIME_SOLVER_TIMEOUT_SEC = 120.0

# Local Anaconda/macOS project directory.
# Original local project directory.
PROJECT_DIR = Path("/Users/arian/ROSE-RAG")

# Fall back to the current notebook directory if the explicit path does not exist.
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "rose_rag_monte_carlo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# Remove raster figures left by any earlier version.
for old_png in OUTPUT_DIR.glob("*.png"):
    old_png.unlink()


def save_vector_figure(figure, stem: str) -> None:
    """Save a figure only in publication-quality vector formats."""
    figure.savefig(
        OUTPUT_DIR / f"{stem}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    figure.savefig(
        OUTPUT_DIR / f"{stem}.svg",
        format="svg",
        bbox_inches="tight",
    )



# RANDOM INSTANCE GENERATION


def generate_random_sources(
    rng: np.random.Generator,
    n_sources: int,
) -> list[SourceSpec]:
    """
    Generate heterogeneous informative binary sources.

    The informativeness condition alpha - beta >= 0.15 is enforced.
    """
    sources: list[SourceSpec] = []

    for source_index in range(n_sources):
        for _ in range(10_000):
            alpha = float(rng.uniform(0.60, 0.95))
            beta = float(rng.uniform(0.05, 0.40))

            if alpha - beta >= 0.15:
                break
        else:
            raise RuntimeError("Could not generate an informative source.")

        cost = float(rng.uniform(0.005, 0.10))

        sources.append(
            SourceSpec(
                alpha=alpha,
                beta=beta,
                cost=cost,
                name=f"S{source_index + 1}",
            )
        )

    return sources


def build_nested_coupled_scenarios(
    sources: list[SourceSpec],
    rho: float,
    rho_grid: np.ndarray,
    eps: float = 1e-9,
):
    """
    Construct a nested finite coupled ambiguity family.

    U(rho_1) is a subset of U(rho_2) whenever rho_1 <= rho_2 because
    every perturbation level up to the requested radius is retained.

    The coupled regimes move all sources coherently, while Rect-ROSE
    later combines source-wise projections arbitrarily.
    """
    if rho < -1e-15:
        raise ValueError("rho must be nonnegative.")

    active_levels = [
        float(level)
        for level in rho_grid
        if level > 0 and level <= rho + 1e-12
    ]

    regimes: list[tuple[float, float, str]] = [(0.0, 0.0, "nominal")]

    for level in active_levels:
        regimes.extend(
            [
                (-level, +level, f"pessimistic_{level:.3f}"),
                (+level, -level, f"optimistic_{level:.3f}"),
                (-level, -level, f"low_rate_{level:.3f}"),
                (+level, +level, f"high_rate_{level:.3f}"),
            ]
        )

    raw_scenarios: list[np.ndarray] = []
    names: list[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario_channels = []

        for source in sources:
            alpha = float(
                np.clip(source.alpha + delta_alpha, eps, 1.0 - eps)
            )
            beta = float(
                np.clip(source.beta + delta_beta, eps, 1.0 - eps)
            )
            scenario_channels.append(make_channel(alpha, beta))

        raw_scenarios.append(np.stack(scenario_channels))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(
        np.stack(raw_scenarios),
        names,
    )

    source_projection_points: list[list[tuple[float, float]]] = []

    for source_index in range(len(sources)):
        points: list[tuple[float, float]] = []

        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (
                float(channel[1, 1]),  # alpha
                float(channel[0, 1]),  # beta
            )

            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)

        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return (
        scenario_channels,
        source_projection_points,
        nominal_channels,
        names,
    )


def classify_action(action: str) -> str:
    if action.startswith("query_"):
        return "query"
    if action == "abstain":
        return "abstain"
    return "answer"


def action_transition(previous: str, current: str) -> str:
    if previous == current:
        return "same_action"

    previous_class = classify_action(previous)
    current_class = classify_action(current)

    if previous_class == current_class == "query":
        return "source_switch"

    return f"{previous_class}_to_{current_class}"



# ONE RANDOM INSTANCE


def solve_random_instance(
    instance_id: int,
    rng: np.random.Generator,
    n_sources: int = N_SOURCES,
):
    prior_y1 = float(rng.uniform(0.05, 0.95))
    abstention_cost = float(rng.uniform(0.15, 0.45))
    sources = generate_random_sources(rng, n_sources)
    costs = np.array([source.cost for source in sources], dtype=float)

    instance_rows: list[dict[str, Any]] = []
    rvoe_rows: list[dict[str, Any]] = []

    for rho in RHO_GRID:
        (
            scenario_channels,
            source_projection_points,
            nominal_channels,
            scenario_names,
        ) = build_nested_coupled_scenarios(
            sources,
            float(rho),
            RHO_GRID,
        )

        start = time.perf_counter()
        nominal_root, _ = nominal_dp(
            prior_y1,
            nominal_channels,
            costs,
            abstention_cost,
        )
        runtime_nominal = time.perf_counter() - start

        start = time.perf_counter()
        (
            static_value,
            _,
            _,
            _,
            static_history,
        ) = static_double_oracle(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            tolerance=STATIC_SOLVER_TOL,
        )
        runtime_static = time.perf_counter() - start

        start = time.perf_counter()
        hcr_root, _ = hcr_dp(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
        )
        runtime_hcr = time.perf_counter() - start

        start = time.perf_counter()
        rectangular_root, _ = rectangular_dp(
            prior_y1,
            source_projection_points,
            costs,
            abstention_cost,
        )
        runtime_rect = time.perf_counter() - start

        nominal_value = float(nominal_root[0])
        hcr_value = float(hcr_root[0])
        rectangular_value = float(rectangular_root[0])
        stopping_value = float(min(hcr_root[3]))

        hierarchy_valid = bool(
            nominal_value <= static_value + ORDER_TOL
            and static_value <= hcr_value + ORDER_TOL
            and hcr_value <= rectangular_value + ORDER_TOL
        )

        row = {
            "instance_id": instance_id,
            "rho": float(rho),
            "prior_y1": prior_y1,
            "abstention_cost": abstention_cost,
            "n_sources": n_sources,
            "n_coupled_scenarios": len(scenario_names),
            "V_nominal": nominal_value,
            "V_static": float(static_value),
            "V_hcr": hcr_value,
            "V_rect": rectangular_value,
            "PoA": float(static_value - nominal_value),
            "PoDC": float(hcr_value - static_value),
            "PoR": float(rectangular_value - hcr_value),
            "action_nominal": nominal_root[1],
            "action_hcr": hcr_root[1],
            "action_rect": rectangular_root[1],
            "stop_value": stopping_value,
            "hierarchy_valid": hierarchy_valid,
            "runtime_nominal_sec": runtime_nominal,
            "runtime_static_sec": runtime_static,
            "runtime_hcr_sec": runtime_hcr,
            "runtime_rect_sec": runtime_rect,
            "static_iterations": len(static_history),
        }

        for source_index, source in enumerate(sources):
            row[f"source_{source_index + 1}_alpha"] = source.alpha
            row[f"source_{source_index + 1}_beta"] = source.beta
            row[f"source_{source_index + 1}_cost"] = source.cost
            row[f"source_{source_index + 1}_informativeness"] = (
                source.alpha - source.beta
            )

            continuation_value = float(
                hcr_root[2][source_index]
            )
            rvoe_rows.append(
                {
                    "instance_id": instance_id,
                    "rho": float(rho),
                    "source_index": source_index,
                    "source": source.name,
                    "RVoE_HCR": stopping_value - continuation_value,
                    "continuation_value": continuation_value,
                    "stop_value": stopping_value,
                    "source_alpha": source.alpha,
                    "source_beta": source.beta,
                    "source_cost": source.cost,
                    "source_informativeness": source.alpha - source.beta,
                }
            )

        instance_rows.append(row)

    # Certificates are checked at one prespecified radius.
    (
        cert_scenarios,
        _,
        _,
        _,
    ) = build_nested_coupled_scenarios(
        sources,
        CERTIFICATE_RHO,
        RHO_GRID,
    )

    certificate_df = root_acquisition_certificates(
        prior_y1,
        cert_scenarios,
        costs,
        abstention_cost,
    )

    certificate_df["instance_id"] = instance_id
    certificate_df["rho"] = CERTIFICATE_RHO
    certificate_df["prior_y1"] = prior_y1
    certificate_df["abstention_cost"] = abstention_cost

    for source_index, source in enumerate(sources):
        mask = certificate_df["source_index"] == source_index
        certificate_df.loc[mask, "source"] = source.name
        certificate_df.loc[mask, "source_alpha"] = source.alpha
        certificate_df.loc[mask, "source_beta"] = source.beta
        certificate_df.loc[mask, "source_cost"] = source.cost

    return (
        pd.DataFrame(instance_rows),
        pd.DataFrame(rvoe_rows),
        certificate_df,
    )



# MONOTONICITY AND TRANSITIONS


def evaluate_instance_level_checks(
    results_df: pd.DataFrame,
    rvoe_df: pd.DataFrame,
):
    monotonicity_rows: list[dict[str, Any]] = []
    transition_rows: list[dict[str, Any]] = []

    for instance_id, group in results_df.groupby("instance_id"):
        group = group.sort_values("rho")

        risk_differences = np.diff(group["V_hcr"].to_numpy())
        static_differences = np.diff(group["V_static"].to_numpy())
        rect_differences = np.diff(group["V_rect"].to_numpy())

        risk_monotone_hcr = bool(
            np.all(risk_differences >= -MONOTONICITY_TOL)
        )
        risk_monotone_static = bool(
            np.all(static_differences >= -MONOTONICITY_TOL)
        )
        risk_monotone_rect = bool(
            np.all(rect_differences >= -MONOTONICITY_TOL)
        )

        instance_rvoe = rvoe_df[
            rvoe_df["instance_id"] == instance_id
        ]

        source_checks: list[bool] = []
        maximum_rvoe_increase = 0.0

        for _, source_group in instance_rvoe.groupby("source_index"):
            source_group = source_group.sort_values("rho")
            rvoe_differences = np.diff(
                source_group["RVoE_HCR"].to_numpy()
            )
            source_checks.append(
                bool(
                    np.all(
                        rvoe_differences
                        <= MONOTONICITY_TOL
                    )
                )
            )
            if len(rvoe_differences):
                maximum_rvoe_increase = max(
                    maximum_rvoe_increase,
                    float(np.max(rvoe_differences)),
                )

        monotonicity_rows.append(
            {
                "instance_id": instance_id,
                "static_risk_monotone": risk_monotone_static,
                "hcr_risk_monotone": risk_monotone_hcr,
                "rect_risk_monotone": risk_monotone_rect,
                "all_source_rvoe_monotone": bool(all(source_checks)),
                "min_hcr_risk_increment": (
                    float(np.min(risk_differences))
                    if len(risk_differences)
                    else np.nan
                ),
                "max_rvoe_increase": maximum_rvoe_increase,
            }
        )

        actions = group["action_hcr"].tolist()
        rhos = group["rho"].tolist()

        for index in range(1, len(actions)):
            transition_rows.append(
                {
                    "instance_id": instance_id,
                    "rho_from": rhos[index - 1],
                    "rho_to": rhos[index],
                    "action_from": actions[index - 1],
                    "action_to": actions[index],
                    "transition": action_transition(
                        actions[index - 1],
                        actions[index],
                    ),
                }
            )

    return (
        pd.DataFrame(monotonicity_rows),
        pd.DataFrame(transition_rows),
    )



# RUNTIME SCALING



def run_runtime_scaling(
    rng: np.random.Generator,
) -> pd.DataFrame:
    """
    Run a guarded runtime-scaling study.

    Exact Rect-ROSE is deliberately censored above
    RECT_EXACT_MAX_SOURCES because its projected belief-set state grows
    combinatorially. A skipped or timed-out solve is recorded explicitly
    in the output instead of blocking the complete experiment.
    """
    rows: list[dict[str, Any]] = []

    if not RUN_RUNTIME_SCALING:
        return pd.DataFrame(rows)

    runtime_checkpoint_path = (
        OUTPUT_DIR / "checkpoint_runtime_scaling.csv"
    )

    for n_sources in RUNTIME_SOURCE_COUNTS:
        for repetition in range(RUNTIME_REPETITIONS):
            prior_y1 = float(rng.uniform(0.10, 0.90))
            abstention_cost = float(rng.uniform(0.20, 0.40))
            sources = generate_random_sources(rng, n_sources)
            costs = np.array(
                [source.cost for source in sources],
                dtype=float,
            )

            (
                scenarios,
                projections,
                nominal_channels,
                names,
            ) = build_nested_coupled_scenarios(
                sources,
                RUNTIME_RHO,
                RHO_GRID,
            )

            method_calls = [
                (
                    "nominal",
                    lambda: nominal_dp(
                        prior_y1,
                        nominal_channels,
                        costs,
                        abstention_cost,
                    ),
                ),
                (
                    "static",
                    lambda: static_double_oracle(
                        prior_y1,
                        scenarios,
                        costs,
                        abstention_cost,
                        tolerance=STATIC_SOLVER_TOL,
                    ),
                ),
                (
                    "hcr",
                    lambda: hcr_dp(
                        prior_y1,
                        scenarios,
                        costs,
                        abstention_cost,
                    ),
                ),
                (
                    "rect",
                    lambda: rectangular_dp(
                        prior_y1,
                        projections,
                        costs,
                        abstention_cost,
                    ),
                ),
            ]

            for method, method_call in method_calls:
                base_row = {
                    "n_sources": n_sources,
                    "repetition": repetition,
                    "method": method,
                    "n_scenarios": len(names),
                    "runtime_limit_sec": (
                        RUNTIME_SOLVER_TIMEOUT_SEC
                    ),
                }

                if (
                    method == "rect"
                    and n_sources > RECT_EXACT_MAX_SOURCES
                ):
                    rows.append(
                        {
                            **base_row,
                            "runtime_sec": np.nan,
                            "status": (
                                "skipped_exact_state_explosion"
                            ),
                        }
                    )
                    print(
                        "Runtime scaling: "
                        f"sources={n_sources}, "
                        f"repetition={repetition + 1}/"
                        f"{RUNTIME_REPETITIONS}, "
                        f"method={method} -> SKIPPED "
                        f"(exact limit={RECT_EXACT_MAX_SOURCES})"
                    )
                    continue

                print(
                    "Runtime scaling: "
                    f"sources={n_sources}, "
                    f"repetition={repetition + 1}/"
                    f"{RUNTIME_REPETITIONS}, "
                    f"method={method} -> START"
                )

                start_time = time.perf_counter()

                try:
                    with solver_time_limit(
                        RUNTIME_SOLVER_TIMEOUT_SEC
                    ):
                        method_call()

                    runtime = time.perf_counter() - start_time
                    status = "completed"

                    print(
                        "Runtime scaling: "
                        f"sources={n_sources}, "
                        f"repetition={repetition + 1}/"
                        f"{RUNTIME_REPETITIONS}, "
                        f"method={method} -> DONE "
                        f"({runtime:.4f} sec)"
                    )

                except SolverTimeoutError:
                    runtime = np.nan
                    status = "timed_out"

                    print(
                        "Runtime scaling: "
                        f"sources={n_sources}, "
                        f"repetition={repetition + 1}/"
                        f"{RUNTIME_REPETITIONS}, "
                        f"method={method} -> TIMEOUT "
                        f"({RUNTIME_SOLVER_TIMEOUT_SEC:.1f} sec)"
                    )

                except MemoryError:
                    runtime = np.nan
                    status = "memory_error"

                    print(
                        "Runtime scaling: "
                        f"sources={n_sources}, "
                        f"repetition={repetition + 1}/"
                        f"{RUNTIME_REPETITIONS}, "
                        f"method={method} -> MEMORY ERROR"
                    )

                rows.append(
                    {
                        **base_row,
                        "runtime_sec": runtime,
                        "status": status,
                    }
                )

            pd.DataFrame(rows).to_csv(
                runtime_checkpoint_path,
                index=False,
            )

    return pd.DataFrame(rows)



# FIGURES AND TABLES


def make_price_distribution_figure(results_df: pd.DataFrame) -> None:
    long_df = results_df.melt(
        id_vars=["instance_id", "rho"],
        value_vars=["PoA", "PoDC", "PoR"],
        var_name="price_type",
        value_name="price",
    )

    figure = plt.figure(figsize=(8.5, 5.2))

    data = [
        long_df.loc[
            long_df["price_type"] == price_type,
            "price",
        ].to_numpy()
        for price_type in ["PoA", "PoDC", "PoR"]
    ]

    # Version-compatible Matplotlib call:
    # `tick_labels` was introduced in Matplotlib 3.9, while older
    # versions expect `labels`. To support both old and new versions,
    # set tick labels separately.
    box_labels = [
        "Price of\nAmbiguity",
        "Price of Dynamic\nConsistency",
        "Price of\nRectangularization",
    ]

    plt.boxplot(
        data,
        positions=[1, 2, 3],
        showfliers=False,
    )
    plt.xticks(
        [1, 2, 3],
        box_labels,
    )
    plt.ylabel("Risk gap")
    plt.title(
        "Distribution of robustness prices across random instances"
    )
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_price_distributions",
    )
    plt.close(figure)


def make_violation_summary_figure(
    results_df: pd.DataFrame,
    monotonicity_df: pd.DataFrame,
    certificate_df: pd.DataFrame,
) -> pd.DataFrame:
    checks = {
        "Hierarchy": int((~results_df["hierarchy_valid"]).sum()),
        "Static risk\nmonotonicity": int(
            (~monotonicity_df["static_risk_monotone"]).sum()
        ),
        "HCR risk\nmonotonicity": int(
            (~monotonicity_df["hcr_risk_monotone"]).sum()
        ),
        "Rect risk\nmonotonicity": int(
            (~monotonicity_df["rect_risk_monotone"]).sum()
        ),
        "RVoE\nmonotonicity": int(
            (~monotonicity_df["all_source_rvoe_monotone"]).sum()
        ),
        "Certificate\nbounds": int(
            (~certificate_df["bound_valid"]).sum()
        ),
    }

    summary = pd.DataFrame(
        {
            "check": list(checks.keys()),
            "violation_count": list(checks.values()),
        }
    )

    figure = plt.figure(figsize=(9, 5))
    plt.bar(
        summary["check"],
        summary["violation_count"],
    )
    plt.ylabel("Number of violations")
    plt.title("Numerical theorem-check violations")
    plt.xticks(rotation=20)
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_hierarchy_violation_summary",
    )
    plt.close(figure)

    return summary


def make_action_transition_figure(
    transition_df: pd.DataFrame,
) -> pd.DataFrame:
    counts = (
        transition_df["transition"]
        .value_counts()
        .rename_axis("transition")
        .reset_index(name="count")
    )
    counts["percentage"] = (
        100.0 * counts["count"] / counts["count"].sum()
    )

    figure = plt.figure(figsize=(9, 5))
    plt.bar(
        counts["transition"],
        counts["percentage"],
    )
    plt.ylabel("Share of adjacent-radius transitions (%)")
    plt.title("Frequency of HCR policy transitions as ambiguity grows")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_action_transition_frequency",
    )
    plt.close(figure)

    return counts



def make_runtime_scaling_figure(
    runtime_df: pd.DataFrame,
) -> None:
    if runtime_df.empty:
        return

    completed = runtime_df.loc[
        runtime_df["status"] == "completed"
    ].dropna(subset=["runtime_sec"])

    if completed.empty:
        return

    summary = (
        completed.groupby(
            ["n_sources", "method"],
            as_index=False,
        )["runtime_sec"]
        .median()
    )

    figure = plt.figure(figsize=(8.5, 5.2))

    for method, group in summary.groupby("method"):
        group = group.sort_values("n_sources")
        plt.plot(
            group["n_sources"],
            group["runtime_sec"],
            marker="o",
            label=method,
        )

    plt.yscale("log")
    plt.xlabel("Number of sources")
    plt.ylabel("Median completed runtime (seconds, log scale)")
    plt.title(
        "Runtime scaling of exact ROSE-RAG formulations"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_runtime_scaling",
    )
    plt.close(figure)


def make_mean_risk_by_radius_figure(
    results_df: pd.DataFrame,
) -> None:
    summary = (
        results_df.groupby("rho", as_index=False)[
            ["V_nominal", "V_static", "V_hcr", "V_rect"]
        ]
        .mean()
    )

    figure = plt.figure(figsize=(8.5, 5.2))

    for column, label in [
        ("V_nominal", "Nominal"),
        ("V_static", "Static-ROSE"),
        ("V_hcr", "HCR-ROSE"),
        ("V_rect", "Rect-ROSE"),
    ]:
        plt.plot(
            summary["rho"],
            summary[column],
            marker="o",
            label=label,
        )

    plt.xlabel("Ambiguity radius")
    plt.ylabel("Mean optimal risk")
    plt.title("Mean robustness hierarchy across random instances")
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_mean_hierarchy_monte_carlo",
    )
    plt.close(figure)


def build_summary_table(
    results_df: pd.DataFrame,
    monotonicity_df: pd.DataFrame,
    certificate_df: pd.DataFrame,
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "metric": "Random instances",
                "value": results_df["instance_id"].nunique(),
            },
            {
                "metric": "Instance-radius evaluations",
                "value": len(results_df),
            },
            {
                "metric": "Hierarchy violations",
                "value": int((~results_df["hierarchy_valid"]).sum()),
            },
            {
                "metric": "HCR risk monotonicity violations",
                "value": int(
                    (~monotonicity_df["hcr_risk_monotone"]).sum()
                ),
            },
            {
                "metric": "RVoE monotonicity violations",
                "value": int(
                    (
                        ~monotonicity_df[
                            "all_source_rvoe_monotone"
                        ]
                    ).sum()
                ),
            },
            {
                "metric": "Certificate bound violations",
                "value": int((~certificate_df["bound_valid"]).sum()),
            },
            {
                "metric": "Mean Price of Ambiguity",
                "value": float(results_df["PoA"].mean()),
            },
            {
                "metric": "Mean Price of Dynamic Consistency",
                "value": float(results_df["PoDC"].mean()),
            },
            {
                "metric": "Mean Price of Rectangularization",
                "value": float(results_df["PoR"].mean()),
            },
            {
                "metric": "No-harm certificate rate",
                "value": float(
                    certificate_df["no_harm_certificate"].mean()
                ),
            },
            {
                "metric": "Rejection certificate rate",
                "value": float(
                    certificate_df["rejection_certificate"].mean()
                ),
            },
        ]
    )



# MAIN EXECUTION



def finalize_result_frames(
    results_df: pd.DataFrame,
    rvoe_df: pd.DataFrame,
    certificate_df: pd.DataFrame,
    *,
    run_runtime_scaling_now: bool,
    rng: np.random.Generator | None = None,
):
    """
    Create final CSVs and figures from already-computed Monte Carlo frames.

    Primary theorem-check outputs are saved before the optional runtime
    scaling stage. Thus a runtime timeout cannot destroy the main study.
    """
    monotonicity_df, transition_df = (
        evaluate_instance_level_checks(
            results_df,
            rvoe_df,
        )
    )

    hierarchy_violations_df = results_df[
        ~results_df["hierarchy_valid"]
    ].copy()

    certificate_violations_df = certificate_df[
        ~certificate_df["bound_valid"]
    ].copy()

    prices_df = results_df[
        [
            "instance_id",
            "rho",
            "PoA",
            "PoDC",
            "PoR",
        ]
    ].copy()

    violation_summary_df = make_violation_summary_figure(
        results_df,
        monotonicity_df,
        certificate_df,
    )
    transition_counts_df = make_action_transition_figure(
        transition_df
    )

    make_price_distribution_figure(results_df)
    make_mean_risk_by_radius_figure(results_df)

    summary_df = build_summary_table(
        results_df,
        monotonicity_df,
        certificate_df,
    )

    # Save the primary study before starting runtime scaling.
    results_df.to_csv(
        OUTPUT_DIR / "results_random_instances.csv",
        index=False,
    )
    hierarchy_violations_df.to_csv(
        OUTPUT_DIR / "results_hierarchy_violations.csv",
        index=False,
    )
    certificate_df.to_csv(
        OUTPUT_DIR / "results_certificate_soundness.csv",
        index=False,
    )
    certificate_violations_df.to_csv(
        OUTPUT_DIR / "results_certificate_violations.csv",
        index=False,
    )
    monotonicity_df.to_csv(
        OUTPUT_DIR / "results_monotonicity.csv",
        index=False,
    )
    prices_df.to_csv(
        OUTPUT_DIR / "results_robustness_prices.csv",
        index=False,
    )
    rvoe_df.to_csv(
        OUTPUT_DIR / "results_rvoe_monte_carlo.csv",
        index=False,
    )
    transition_df.to_csv(
        OUTPUT_DIR / "results_action_transitions.csv",
        index=False,
    )
    transition_counts_df.to_csv(
        OUTPUT_DIR / "results_action_transition_summary.csv",
        index=False,
    )
    violation_summary_df.to_csv(
        OUTPUT_DIR / "results_violation_summary.csv",
        index=False,
    )
    summary_df.to_csv(
        OUTPUT_DIR / "results_monte_carlo_summary.csv",
        index=False,
    )

    metadata = {
        "seed": SEED,
        "n_instances": int(
            results_df["instance_id"].nunique()
        ),
        "rho_grid": RHO_GRID.tolist(),
        "certificate_rho": CERTIFICATE_RHO,
        "n_sources": N_SOURCES,
        "order_tolerance": ORDER_TOL,
        "monotonicity_tolerance": MONOTONICITY_TOL,
        "static_solver_tolerance": STATIC_SOLVER_TOL,
        "runtime_scaling_requested": bool(
            run_runtime_scaling_now
        ),
        "runtime_source_counts": RUNTIME_SOURCE_COUNTS,
        "runtime_repetitions": RUNTIME_REPETITIONS,
        "rect_exact_max_sources": RECT_EXACT_MAX_SOURCES,
        "runtime_solver_timeout_sec": (
            RUNTIME_SOLVER_TIMEOUT_SEC
        ),
    }

    with (
        OUTPUT_DIR / "experiment_metadata.json"
    ).open("w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    print(
        "\nPrimary Monte Carlo results were saved successfully "
        "before runtime scaling."
    )
    print(summary_df.to_string(index=False))

    runtime_df = pd.DataFrame()

    if run_runtime_scaling_now:
        if rng is None:
            rng = np.random.default_rng(SEED + 1_000_000)

        runtime_df = run_runtime_scaling(rng)
        runtime_df.to_csv(
            OUTPUT_DIR / "results_runtime_scaling.csv",
            index=False,
        )
        make_runtime_scaling_figure(runtime_df)

        print(
            "\nSafe runtime-scaling stage completed. "
            "Skipped/time-limited exact solves are recorded in "
            "the status column."
        )

    return {
        "results": results_df,
        "rvoe": rvoe_df,
        "certificates": certificate_df,
        "monotonicity": monotonicity_df,
        "transitions": transition_df,
        "runtime": runtime_df,
        "summary": summary_df,
    }


def recover_from_checkpoints(
    *,
    run_runtime_scaling_now: bool = False,
):
    """
    Recover the completed Monte Carlo study without recomputing instances.

    Use this after interrupting the old notebook at the five-source
    runtime-scaling stage.
    """
    results_path = (
        OUTPUT_DIR / "checkpoint_random_instances.csv"
    )
    rvoe_path = OUTPUT_DIR / "checkpoint_rvoe.csv"
    certificate_path = (
        OUTPUT_DIR / "checkpoint_certificates.csv"
    )

    missing = [
        str(path)
        for path in [
            results_path,
            rvoe_path,
            certificate_path,
        ]
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Checkpoint files were not found:\n"
            + "\n".join(missing)
        )

    results_df = pd.read_csv(results_path)
    rvoe_df = pd.read_csv(rvoe_path)
    certificate_df = pd.read_csv(certificate_path)

    print(
        "Recovered "
        f"{results_df['instance_id'].nunique()} completed "
        "Monte Carlo instances from checkpoints."
    )

    return finalize_result_frames(
        results_df,
        rvoe_df,
        certificate_df,
        run_runtime_scaling_now=run_runtime_scaling_now,
        rng=np.random.default_rng(SEED + 1_000_000),
    )


def run_monte_carlo_experiment(
    *,
    run_runtime_scaling_now: bool | None = None,
):
    if run_runtime_scaling_now is None:
        run_runtime_scaling_now = RUN_RUNTIME_SCALING

    rng = np.random.default_rng(SEED)

    all_results: list[pd.DataFrame] = []
    all_rvoe: list[pd.DataFrame] = []
    all_certificates: list[pd.DataFrame] = []

    total_start = time.perf_counter()

    for instance_id in range(N_INSTANCES):
        result_df, rvoe_df, certificate_df = solve_random_instance(
            instance_id,
            rng,
        )

        all_results.append(result_df)
        all_rvoe.append(rvoe_df)
        all_certificates.append(certificate_df)

        completed = instance_id + 1

        if (
            completed % CHECKPOINT_EVERY == 0
            or completed == N_INSTANCES
        ):
            checkpoint_results = pd.concat(
                all_results,
                ignore_index=True,
            )
            checkpoint_rvoe = pd.concat(
                all_rvoe,
                ignore_index=True,
            )
            checkpoint_certificates = pd.concat(
                all_certificates,
                ignore_index=True,
            )

            checkpoint_results.to_csv(
                OUTPUT_DIR / "checkpoint_random_instances.csv",
                index=False,
            )
            checkpoint_rvoe.to_csv(
                OUTPUT_DIR / "checkpoint_rvoe.csv",
                index=False,
            )
            checkpoint_certificates.to_csv(
                OUTPUT_DIR / "checkpoint_certificates.csv",
                index=False,
            )

            elapsed = time.perf_counter() - total_start
            print(
                f"Completed {completed}/{N_INSTANCES} instances "
                f"(elapsed {elapsed:.2f} sec)"
            )

    results_df = pd.concat(all_results, ignore_index=True)
    rvoe_df = pd.concat(all_rvoe, ignore_index=True)
    certificate_df = pd.concat(
        all_certificates,
        ignore_index=True,
    )

    return finalize_result_frames(
        results_df,
        rvoe_df,
        certificate_df,
        run_runtime_scaling_now=bool(
            run_runtime_scaling_now
        ),
        rng=rng,
    )



## Main experiment

This execution starts independently of earlier checkpoints.


In [ ]:
experiment = run_monte_carlo_experiment(
    run_runtime_scaling_now=False
)

## Plot recovery from saved results

This cell reloads completed results and regenerates plots without solving the instances again.


In [ ]:
plt.close("all")

experiment = recover_from_checkpoints(
    run_runtime_scaling_now=False
)

## Result summaries and diagnostics


In [ ]:
from IPython.display import display

display(experiment["summary"])

print("Hierarchy violations:")
display(
    experiment["results"].loc[
        ~experiment["results"]["hierarchy_valid"]
    ]
)

print("Monotonicity violations:")
display(
    experiment["monotonicity"].loc[
        ~(
            experiment["monotonicity"]["hcr_risk_monotone"]
            & experiment["monotonicity"]["all_source_rvoe_monotone"]
        )
    ]
)

print("Certificate violations:")
display(
    experiment["certificates"].loc[
        ~experiment["certificates"]["bound_valid"]
    ]
)

## Separate runtime-scaling experiment

Runtime scaling uses the completed main results. The configured guard skips exact Rectangular evaluation with five sources.


In [ ]:
runtime_rng = np.random.default_rng(SEED + 1_000_000)

runtime_df = run_runtime_scaling(runtime_rng)

runtime_df.to_csv(
    OUTPUT_DIR / "results_runtime_scaling.csv",
    index=False,
)

make_runtime_scaling_figure(runtime_df)

display(runtime_df)

## Saved vector figures


In [ ]:
from IPython.display import SVG, display

for filename in [
    "figure_mean_hierarchy_monte_carlo.svg",
    "figure_price_distributions.svg",
    "figure_hierarchy_violation_summary.svg",
    "figure_action_transition_frequency.svg",
    "figure_runtime_scaling.svg",
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(filename)
        display(SVG(filename=str(path)))

In [ ]:
def make_violation_summary_figure(
    results_df: pd.DataFrame,
    monotonicity_df: pd.DataFrame,
    certificate_df: pd.DataFrame,
) -> pd.DataFrame:

    checks = {
        "Hierarchy": int(
            (~results_df["hierarchy_valid"]).sum()
        ),
        "Static risk\nmonotonicity": int(
            (~monotonicity_df["static_risk_monotone"]).sum()
        ),
        "HCR risk\nmonotonicity": int(
            (~monotonicity_df["hcr_risk_monotone"]).sum()
        ),
        "Rect risk\nmonotonicity": int(
            (~monotonicity_df["rect_risk_monotone"]).sum()
        ),
        "RVoE\nmonotonicity": int(
            (
                ~monotonicity_df[
                    "all_source_rvoe_monotone"
                ]
            ).sum()
        ),
        "Certificate\nbounds": int(
            (~certificate_df["bound_valid"]).sum()
        ),
    }

    summary = pd.DataFrame(
        {
            "check": list(checks.keys()),
            "violation_count": list(checks.values()),
        }
    )

    figure, ax = plt.subplots(figsize=(10, 5.5))

    x_positions = np.arange(len(summary))
    values = summary["violation_count"].to_numpy()

    ax.bar(
        x_positions,
        values,
        width=0.65,
    )

    # Minimum axis height keeps zero-valued results visible.
    maximum_value = int(values.max())

    if maximum_value == 0:
        upper_limit = 1.0
    else:
        upper_limit = maximum_value * 1.25

    ax.set_ylim(0, upper_limit)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        summary["check"],
        rotation=20,
        ha="right",
    )

    # Annotate the value and status of each check.
    for position, value in zip(x_positions, values):

        if value == 0:
            label = "0 violations\nPASS"
            text_height = upper_limit * 0.08
        else:
            label = f"{value} violations\nCHECK"
            text_height = value + upper_limit * 0.03

        ax.text(
            position,
            text_height,
            label,
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    ax.set_ylabel("Number of violations")
    ax.set_title(
        "Monte Carlo theorem-check summary "
        f"({results_df['instance_id'].nunique()} random instances)"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.35,
    )

    figure.tight_layout()

    save_vector_figure(
        figure,
        "figure_hierarchy_violation_summary",
    )

    plt.close(figure)

    return summary

In [ ]:
violation_summary_df = make_violation_summary_figure(
    experiment["results"],
    experiment["monotonicity"],
    experiment["certificates"],
)

display(violation_summary_df)